Author: MJ

In [41]:
import sys
print(sys.version)

3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]


In [42]:
import os
import glob

raw_dir = "./opcodes"
# raw_dir = "./Raw_Extracted_Files
output_dir = "./ml_data"
os.makedirs(output_dir, exist_ok=True)
files = glob.glob(f"{raw_dir}/*.opcode")
print(f"Found {len(files)} .opcode files")


Found 48 .opcode files


Install

In [43]:
# %pip install scikit-learn pandas numpy --quiet

Imports

In [44]:
import os, glob, pandas as pd, numpy as np, pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report

In [46]:
# raw_dir = "./Raw_Extracted_Files"
raw_dir = "./opcodes"
output_dir = "./ml_data"
os.makedirs(output_dir, exist_ok=True)

files = glob.glob(f"{raw_dir}/*.opcode")
print(f"Found {len(files)} .opcode files")
if len(files) == 0:
    print(" No files! Upload Raw_Extracted_Files/ or run extraction script first.")

Found 48 .opcode files


Process to features

In [ ]:
docs, labels = [], []
for fpath in files:
    with open(fpath, 'r') as f:
        content = f.read().strip()
        if content:
            docs.append(' '.join(content.split('\\n')))
            labels.append('malware') 

print(f"Processed {len(docs)} malware samples")

Processed 47 malware samples


In [47]:
# TF-IDF
vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=2000)
X = vectorizer.fit_transform(docs)
y = pd.Series(labels)

os.makedirs("ml_data", exist_ok=True)
pd.DataFrame.sparse.from_spmatrix(X).to_csv('ml_data/features.csv')
pd.Series(y).to_csv('ml_data/labels.csv', index=False)
pickle.dump((X, y, vectorizer), open('ml_data/processed_data.pkl', 'wb'))
print(f"Saved! {X.shape} features, {len(y)} malware samples")

Saved! (48, 2000) features, 47 malware samples


CSV converts sparse TF-IDF matrices to dense format, causing memory and file sizes to explode, loses exact data types as float sparsity becomes full floats, and read/write operations are 10-80x slower for ML data processing. In contrast, pickle stores sparse X, y Series, and fitted vectorizer as exact Python objects, delivering 80x faster speeds and smaller files while preserving sparsity.

In [ ]:
# df_features = pd.read_csv('ml_data/features.csv')
# df_features.head(5)

In [ ]:
# from scipy import sparse
# X = sparse.load_npz('ml_data/features_sparse.npz')
# y = pd.read_csv('ml_data/labels.csv')
# print(X.nnz)

In [ ]:
# print(f"X: {X.shape}, NNZ: {X.nnz:,}")
# print(f"y: {y.shape}")
# print(f"Sparsity: {100*X.nnz/(X.shape[0]*X.shape[1]):.1f}% non-zero")
# print(f"Features per sample: {X.nnz/X.shape[0]:.0f} on average")

In [49]:
from pathlib import Path

def load_opcodes(raw_dir="./opcodes"):
# def load_opcodes(raw_dir="./Raw_Extracted_Files"):    
    
    docs, labels = [], []
    for filepath in Path(raw_dir).glob("*.opcode"):
        with open(filepath, 'r') as f:
            opcode_seq = ' '.join(line.strip() for line in f)
            docs.append(opcode_seq)
            labels.append(filepath.stem.split('_')[0])  
    return docs, np.array(labels)

# docs, y = load_opcodes("./Raw_Extracted_Files")
docs, y = load_opcodes("./opcodes")
print(f"Loaded {len(docs)} opcode sequences")

vectorizer1 = TfidfVectorizer(ngram_range=(1,1), max_features=1000)
X1 = vectorizer1.fit_transform(docs)  # 1-gram only
print(f"1-gram: {X1.shape}, NNZ: {X1.nnz:,}")

vectorizer2 = TfidfVectorizer(ngram_range=(2,2), max_features=1000)
X2 = vectorizer2.fit_transform(docs)  # 2-gram only
print(f"2-gram: {X2.shape}, NNZ: {X2.nnz:,}")

Loaded 48 opcode sequences
1-gram: (48, 611), NNZ: 4,588
2-gram: (48, 1000), NNZ: 18,678


In [ ]:
pickle.dump((X1, X2, y, vectorizer1, vectorizer2), open('ml_data/preprocessing_data.pkl', 'wb'))
print("Pre-processing complete. Data saved to ml_data/preprocessing_data.pkl")

Pre-processing complete. Data saved to ml_data/preprocessing_data.pkl


Data Augumentation Pipeline

In [ ]:
import pickle
import numpy as np
from collections import Counter
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler
from scipy import sparse

# Set reproducibility seed
np.random.seed(42)

# Load original preprocessed data
print("Loading original data...")
with open('ml_data/preprocessing_data.pkl', 'rb') as f:
    X1, X2, y, vectorizer1, vectorizer2 = pickle.load(f)

print(f"Original: {len(y)} samples, {len(np.unique(y))} classes")
print("Original distribution:", Counter(y))

def augment_class(X_class, y_class, target_samples=10):
    """Augment each class to target_samples using bootstrap resampling + small noise.
       Works for both dense and sparse feature matrices.
    """
    current_count = len(y_class)
    if current_count >= target_samples:
        return X_class, y_class

    n_repeats = target_samples // current_count
    remainder = target_samples % current_count

    augmented_X = []
    augmented_y = []

    # Bootstrap resampling
    for _ in range(n_repeats + 1):
        X_boot, y_boot = resample(X_class, y_class, random_state=np.random.randint(10000))
        augmented_X.append(X_boot)
        augmented_y.append(y_boot)

    # Sparse-aware noise addition
    if remainder > 0:
        if sparse.issparse(X_class):
            noise = sparse.random(X_class.shape[0], X_class.shape[1], density=0.1, 
                                  data_rvs=lambda s: np.random.normal(1.0, 0.03, size=s))
            noise_X = X_class.multiply(noise)
            noise_X = noise_X[:remainder]
        else:
            noise_X = X_class + np.random.normal(0, 0.03, X_class.shape) * (X_class > 0)
            noise_X = noise_X[:remainder]
        augmented_X.append(noise_X)
        augmented_y.append(np.repeat(y_class[0], remainder))

    if sparse.issparse(X_class):
        X_out = sparse.vstack(augmented_X)
    else:
        X_out = np.vstack(augmented_X)

    y_out = np.hstack(augmented_y)
    return X_out, y_out

# Process each class
print("Augmenting each class to 10 samples...")
unique_classes = np.unique(y)

X1_augmented_list = []
X2_augmented_list = []
y_augmented_list = []

for cls in unique_classes:
    print(f"Processing class '{cls}'...")
    mask = y == cls
    X1_cls, X2_cls, y_cls = X1[mask], X2[mask], y[mask]

    X1_aug, y_aug = augment_class(X1_cls, y_cls, target_samples=10)
    X2_aug, _ = augment_class(X2_cls, y_cls, target_samples=10)

    X1_augmented_list.append(X1_aug)
    X2_augmented_list.append(X2_aug)
    y_augmented_list.append(y_aug)

# Combine
if sparse.issparse(X1_augmented_list[0]):
    X1_final = sparse.vstack(X1_augmented_list)
    X2_final = sparse.vstack(X2_augmented_list)
else:
    X1_final = np.vstack(X1_augmented_list)
    X2_final = np.vstack(X2_augmented_list)

y_final = np.hstack(y_augmented_list)

print(f"Augmentation complete: {len(y_final)} samples ({len(np.unique(y_final))} classes)")
print("New distribution:", Counter(y_final))
print(f"Samples per class: {len(y_final)//len(np.unique(y_final))} avg")

# Scaling
print("Applying scaling (sparse-safe)...")
scaler1 = StandardScaler(with_mean=False)
scaler2 = StandardScaler(with_mean=False)
X1_final = scaler1.fit_transform(X1_final)
X2_final = scaler2.fit_transform(X2_final)

# Save result
output_file = 'ml_data/manual_augmented_data_sparse.pkl'
with open(output_file, 'wb') as f:
    pickle.dump((X1_final, X2_final, y_final, scaler1, scaler2), f)

print(f"Data saved as '{output_file}'")
print("Ready for classification with realistic, balanced samples.")


Loading original data...
Original: 48 samples, 33 classes
Original distribution: Counter({np.str_('APT29'): 3, np.str_('APT1'): 2, np.str_('APT28'): 2, np.str_('APT33'): 2, np.str_('APT39'): 2, np.str_('APT41'): 2, np.str_('Cobalt'): 2, np.str_('FIN5'): 2, np.str_('FIN7'): 2, np.str_('Moafee'): 2, np.str_('Patchwork'): 2, np.str_('Sidewinder'): 2, np.str_('TEMPVeles'): 2, np.str_('Turla'): 2, np.str_('admin338'): 1, np.str_('APT16'): 1, np.str_('APT17'): 1, np.str_('APT19'): 1, np.str_('APT3'): 1, np.str_('Axiom'): 1, np.str_('BlueMockingbird'): 1, np.str_('BRONZEBUTLER'): 1, np.str_('Chimera'): 1, np.str_('DeepPanda'): 1, np.str_('Dragonfly'): 1, np.str_('Elderwood'): 1, np.str_('Evilnum'): 1, np.str_('Gallium'): 1, np.str_('Gamaredon'): 1, np.str_('HAFNIUM'): 1, np.str_('Ke3chang'): 1, np.str_('MenuPass'): 1, np.str_('SandwormTeam'): 1})
Augmenting each class to 10 samples...
Processing class 'APT1'...
Processing class 'APT16'...
Processing class 'APT17'...
Processing class 'APT19'..